# 05 - Intervention Simulator

This notebook provides an **interactive ROI simulator** for churn intervention campaigns:

1. **Campaign Planning**: Define targeting rules and costs
2. **ROI Simulation**: Estimate business impact
3. **Threshold Optimization**: Find optimal risk threshold
4. **Sensitivity Analysis**: Test assumptions
5. **Scenario Comparison**: Compare multiple strategies

**Use Case**: A marketing team wants to know:
- How many customers to target?
- What's the expected ROI of a retention campaign?
- How do results change with different assumptions?

In [ ]:
# Imports
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

from src.simulator import CampaignParams, InterventionSimulator, quick_roi

# Try to import widgets
try:
    import ipywidgets as widgets
    from IPython.display import HTML, Markdown, display
    from ipywidgets import fixed, interact, interactive
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed. Run: pip install ipywidgets")

# Settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Predictions

In [ ]:
# Load predictions and segments
predictions = pd.read_parquet('../data/features/churn_predictions.parquet')
segmented = pd.read_parquet('../data/features/segmented_customers.parquet')

# Merge
df = segmented.merge(predictions[['visitorid', 'churn_proba']], on='visitorid', suffixes=('', '_y'))
df = df.drop(columns=[c for c in df.columns if c.endswith('_y')])

print(f"Loaded {len(df):,} customers")
print(f"Avg churn probability: {df['churn_proba'].mean():.1%}")
print(f"Segments: {df['segment'].nunique()}")

## 2. Quick ROI Estimate

Before diving into details, let's do a quick back-of-envelope calculation.

In [ ]:
# Quick estimate: Target top 20% at-risk customers
top_20_threshold = df['churn_proba'].quantile(0.8)
top_20 = df[df['churn_proba'] >= top_20_threshold]

quick_estimate = quick_roi(
    n=len(top_20),
    churn_rate=top_20['churn_proba'].mean(),
    ltv=100,  # Assumed average LTV
    cost=5,
    lift=0.20,
    response=0.30
)

print("Quick ROI Estimate (Top 20% At-Risk):")
print("-" * 40)
for k, v in quick_estimate.items():
    print(f"{k}: {v}")

## 3. Interactive Campaign Simulator

Use the sliders below to adjust campaign parameters and see ROI impact in real-time.

In [ ]:
# Initialize simulator with base assumptions
simulator = InterventionSimulator(ltv=100)

In [ ]:
if HAS_WIDGETS:
    def run_simulation(risk_threshold, cost, lift, ltv, response):
        """Run simulation with given parameters and display results.
        """
        # Update simulator LTV
        simulator.ltv = ltv

        # Create campaign
        campaign = CampaignParams(
            name="Interactive Campaign",
            cost_per_contact=cost,
            discount=10,
            lift=lift / 100,
            response_rate=response / 100
        )

        # Run simulation
        result = simulator.run(
            probs=df['churn_proba'],
            campaign=campaign,
            threshold=risk_threshold / 100
        )

        # Display results
        display(HTML(f"""
        <h3>Campaign Results</h3>
        <table style='font-size: 14px;'>
        <tr><td><b>Targeted Customers:</b></td><td>{result.n_targeted:,} ({result.pct_targeted:.1%})</td></tr>
        <tr><td><b>Expected Churners (Baseline):</b></td><td>{result.churners_baseline:,}</td></tr>
        <tr><td><b>Expected Saves:</b></td><td>{result.saves:,}</td></tr>
        <tr><td><b>Total Campaign Cost:</b></td><td>${result.total_cost:,.0f}</td></tr>
        <tr><td><b>Revenue from Saves:</b></td><td>${result.saves_rev:,.0f}</td></tr>
        <tr><td><b>Incremental Revenue:</b></td><td style='color: {'green' if result.inc_rev > 0 else 'red'}'>
            ${result.inc_rev:,.0f}</td></tr>
        <tr><td><b>ROI:</b></td><td style='font-size: 18px; font-weight: bold; color: {'green' if result.roi > 0 else 'red'}'>
            {result.roi:.0%}</td></tr>
        <tr><td><b>Cost per Customer Saved:</b></td><td>${result.cps:.2f}</td></tr>
        </table>
        """))

        # ROI gauge visualization
        fig, ax = plt.subplots(figsize=(8, 3))
        roi_color = 'green' if result.roi > 0 else 'red'
        ax.barh(['ROI'], [min(result.roi * 100, 500)], color=roi_color, height=0.5)
        ax.axvline(x=0, color='black', linewidth=2)
        ax.set_xlim(-200, 500)
        ax.set_xlabel('ROI (%)')
        ax.set_title(f'Campaign ROI: {result.roi:.0%}')
        plt.tight_layout()
        plt.show()

        return result

    # Create interactive widget
    interactive_sim = interactive(
        run_simulation,
        risk_threshold=widgets.IntSlider(
            value=50, min=10, max=90, step=5,
            description='Risk Threshold (%):',
            style={'description_width': '150px'},
            layout=widgets.Layout(width='500px')
        ),
        cost=widgets.FloatSlider(
            value=5, min=1, max=20, step=1,
            description='Cost per Contact ($):',
            style={'description_width': '150px'},
            layout=widgets.Layout(width='500px')
        ),
        lift=widgets.IntSlider(
            value=20, min=5, max=50, step=5,
            description='Expected Lift (%):',
            style={'description_width': '150px'},
            layout=widgets.Layout(width='500px')
        ),
        ltv=widgets.IntSlider(
            value=100, min=50, max=300, step=25,
            description='Avg Customer LTV ($):',
            style={'description_width': '150px'},
            layout=widgets.Layout(width='500px')
        ),
        response=widgets.IntSlider(
            value=30, min=10, max=60, step=5,
            description='Response Rate (%):',
            style={'description_width': '150px'},
            layout=widgets.Layout(width='500px')
        )
    )

    display(interactive_sim)
else:
    print("Widgets not available. Running with default parameters...")
    campaign = CampaignParams(cost=5, lift=0.20, response=0.30)
    result = simulator.run(df['churn_proba'], campaign, threshold=0.5)
    print(result.summary())

## 4. Threshold Optimization

Find the optimal risk threshold that maximizes ROI.

In [ ]:
# Optimize threshold
campaign = CampaignParams(cost_per_contact=5, lift=0.20, response_rate=0.30)
threshold_analysis = simulator.optimize(
    probs=df['churn_proba'],
    campaign=campaign,
    thresholds=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
)

print("Threshold Analysis:")
threshold_analysis

In [ ]:
# Visualize threshold trade-offs
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ROI by threshold
colors = ['green' if x > 0 else 'red' for x in threshold_analysis['roi']]
axes[0].bar(threshold_analysis['thresh'].astype(str), threshold_analysis['roi'] * 100, color=colors)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('Risk Threshold')
axes[0].set_ylabel('ROI (%)')
axes[0].set_title('ROI by Threshold')

# Customers targeted vs ROI
axes[1].plot(threshold_analysis['pct'] * 100, threshold_analysis['roi'] * 100, 'bo-')
axes[1].set_xlabel('% Customers Targeted')
axes[1].set_ylabel('ROI (%)')
axes[1].set_title('ROI vs Targeting Breadth')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Expected saves vs cost
axes[2].scatter(threshold_analysis['cost'], threshold_analysis['saves'],
               c=threshold_analysis['roi'], cmap='RdYlGn', s=100)
axes[2].set_xlabel('Total Cost ($)')
axes[2].set_ylabel('Expected Saves')
axes[2].set_title('Saves vs Cost (color = ROI)')

plt.tight_layout()
plt.savefig('../figures/threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify optimal threshold
optimal = threshold_analysis[threshold_analysis['best']]
if len(optimal) > 0:
    print(f"\nOptimal Threshold: {optimal['thresh'].iloc[0]} (ROI: {optimal['roi'].iloc[0]:.0%}")

## 5. Sensitivity Analysis

How sensitive is ROI to key assumptions?

In [ ]:
# Run sensitivity analysis
sensitivity = simulator.sensitivity(
    probs=df['churn_proba'],
    base=CampaignParams(),
    threshold=0.5
)

# Visualize sensitivities
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sensitivity to lift
ax = axes[0]
data = sensitivity['lift']
colors = ['green' if x > 0 else 'red' for x in data['roi']]
ax.bar(data['lift'].astype(str), data['roi'] * 100, color=colors)
ax.set_xlabel('Expected Lift')
ax.set_ylabel('ROI (%)')
ax.set_title('Sensitivity to Campaign Effectiveness')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Sensitivity to cost
ax = axes[1]
data = sensitivity['cost']
colors = ['green' if x > 0 else 'red' for x in data['roi']]
ax.bar(data['cost'].astype(str), data['roi'] * 100, color=colors)
ax.set_xlabel('Cost per Contact ($)')
ax.set_ylabel('ROI (%)')
ax.set_title('Sensitivity to Contact Cost')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Sensitivity to LTV
ax = axes[2]
data = sensitivity['ltv']
colors = ['green' if x > 0 else 'red' for x in data['roi']]
ax.bar(data['ltv'].astype(str), data['roi'] * 100, color=colors)
ax.set_xlabel('Average LTV ($)')
ax.set_ylabel('ROI (%)')
ax.set_title('Sensitivity to Customer LTV')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('../figures/sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Scenario Comparison

Compare different campaign strategies side-by-side.

In [ ]:
# Define scenarios
scenarios = [
    {
        "name": "Aggressive (Top 30%)",
        "top_pct": 30,
        "cost": 5,
        "lift": 0.15,
        "response": 0.25
    },
    {
        "name": "Balanced (Top 20%)",
        "top_pct": 20,
        "cost": 5,
        "lift": 0.20,
        "response": 0.30
    },
    {
        "name": "Conservative (Top 10%)",
        "top_pct": 10,
        "cost": 10,
        "lift": 0.25,
        "response": 0.35
    },
    {
        "name": "High-Touch (Top 5%)",
        "top_pct": 5,
        "cost": 20,
        "lift": 0.35,
        "response": 0.40
    },
]

# Compare scenarios
comparison = simulator.compare(df['churn_proba'], scenarios)
print("Scenario Comparison:")
comparison

In [ ]:
# Visualize scenario comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROI comparison
roi_values = [float(x.rstrip('%')) for x in comparison['roi']]
colors = ['green' if x > 0 else 'red' for x in roi_values]
axes[0].barh(comparison['campaign'], roi_values, color=colors)
axes[0].set_xlabel('ROI (%)')
axes[0].set_title('ROI by Campaign Strategy')
axes[0].axvline(x=0, color='black', linewidth=1)

# Add value labels
for i, (name, roi) in enumerate(zip(comparison['campaign'], roi_values)):
    axes[0].text(roi + 5, i, f'{roi:.0f}%', va='center')

# Incremental revenue comparison
revenue_values = [float(x.replace('$', '').replace(',', '')) for x in comparison['inc_rev']]
colors = ['green' if x > 0 else 'red' for x in revenue_values]
axes[1].barh(comparison['campaign'], revenue_values, color=colors)
axes[1].set_xlabel('Incremental Revenue ($)')
axes[1].set_title('Incremental Revenue by Strategy')
axes[1].axvline(x=0, color='black', linewidth=1)

plt.tight_layout()
plt.savefig('../figures/scenario_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Generate Targeting List

Export a prioritized list of customers to target.

In [ ]:
# Generate targeting list
targeting_list = simulator.targeting_list(
    ids=df['visitorid'],
    probs=df['churn_proba'],
    segments=df['segment'] if 'segment' in df.columns else None,
    threshold=0.5,
    top_n=1000
)

print(f"Generated targeting list with {len(targeting_list):,} customers")
print("\nTop 10 Priority Customers:")
targeting_list.head(10)

In [ ]:
# Priority distribution
print("\nPriority Distribution:")
print(targeting_list['priority'].value_counts())

In [ ]:
# Save targeting list
targeting_list.to_csv('../data/processed/targeting_list.csv', index=False)
print("\nSaved targeting list to ../data/processed/targeting_list.csv")

## 8. Executive Summary

### Recommended Campaign Strategy

Based on the analysis:

1. **Target the Top 20% at-risk customers** (Balanced approach)
   - Best trade-off between reach and efficiency
   - Positive ROI under most assumptions

2. **Expected Results:**
   - Customers targeted: ~20% of base
   - Expected saves: Varies by lift achieved
   - Projected ROI: Positive under conservative assumptions

3. **Key Success Factors:**
   - Campaign effectiveness (lift) is the most sensitive parameter
   - Keep contact costs low through automation where possible
   - Focus on high-value segments for maximum impact

### Next Steps
1. Run A/B test to validate expected lift
2. Start with "Cannot Lose Them" and "At Risk" segments
3. Monitor and iterate based on actual results